In [1]:
# Install the datasets library if you haven't already
!pip install datasets

from datasets import load_dataset

dataset_name = "BW/RU_SPELLCHECK_DEVICE"

try:
    # Load the dataset from Hugging Face
    dataset = load_dataset(dataset_name)
    print(f"Successfully loaded dataset: {dataset_name}")

    # Display information about the dataset
    print("\nDataset Structure:")
    print(dataset)

    # If the dataset has splits (e.g., 'train', 'validation', 'test'), you can access them
    # For example, to show the first few examples of the 'train' split:
    if 'train' in dataset:
        print("\nFirst 5 examples from the 'train' split:")
        for i in range(min(5, len(dataset['train']))):
            print(dataset['train'][i])
    elif len(dataset) > 0:
        # If no 'train' split, show from the first available split
        first_split_name = list(dataset.keys())[0]
        print(f"\nFirst 5 examples from the '{first_split_name}' split:")
        for i in range(min(5, len(dataset[first_split_name]))):
            print(dataset[first_split_name][i])

except Exception as e:
    print(f"Error loading the dataset '{dataset_name}': {e}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/598 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/69.4k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/67.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/296 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/296 [00:00<?, ? examples/s]

Successfully loaded dataset: BW/RU_SPELLCHECK_DEVICE

Dataset Structure:
DatasetDict({
    train: Dataset({
        features: ['id', 'created_at', 'player_id', 'device_type', 'original', 'typed', '__index_level_0__'],
        num_rows: 296
    })
    test: Dataset({
        features: ['id', 'created_at', 'player_id', 'device_type', 'original', 'typed', '__index_level_0__'],
        num_rows: 296
    })
})

First 5 examples from the 'train' split:
{'id': 'd7c4d975-656f-49bf-b58d-92197a55a12f', 'created_at': '2026-05-28 21:18:35.78+00', 'player_id': '3f42c4c3-0fbb-4635-81d7-f8a629214b7f', 'device_type': 'desktop', 'original': 'Две молодые девушки вели ее под руки.', 'typed': 'Две молодые демушки вели ее под рукию', '__index_level_0__': 501}
{'id': '3839679d-9451-42d0-bfa1-75c0c8908ded', 'created_at': '2026-05-29 17:45:50.286+00', 'player_id': 'f46007cc-49de-48e3-b595-3e38ef1b2d37', 'device_type': 'mobile', 'original': 'В эту ночь явилась ко мне покойница баронесса фон-В.', 'typed': 'в эт

In [3]:
!pip install transformers accelerate evaluate scikit-learn -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00


In [9]:
!pip install torchvision

In [15]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import DatasetDict, Dataset, ClassLabel, Value
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import evaluate

# Install necessary libraries


# Check if 'dataset' is available from previous cell
if 'dataset' not in locals():
    raise ValueError("Dataset not found. Please ensure 'dataset' is loaded in a previous cell.")

# --- 1. Prepare the dataset ---
# Ensure the dataset has 'train' and 'test' splits, or create them if only one split exists
if 'train' not in dataset and 'validation' not in dataset and 'test' not in dataset:
    # Assuming the loaded dataset has only one split, we'll use that as 'train' and create a small 'test' split
    print("Splitting the dataset into train and test...")
    dataset = dataset['train'].train_test_split(test_size=0.1, seed=42) # Adjust split if needed
elif 'train' not in dataset:
    raise ValueError("No 'train' split found in the dataset. Please ensure your dataset has a 'train' split.")

# Get unique device_type labels and create mappings
print("Preparing labels...")
labels = dataset['train'].unique('device_type')
labels = sorted(labels) # Ensure consistent order
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}
num_labels = len(labels)

print(f"Detected labels: {labels}")
print(f"Label to ID mapping: {label2id}")

# --- 2. Load pre-trained BERT model and tokenizer ---
model_name = "DeepPavlov/rubert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # Ensure 'typed' is treated as string for tokenization
    return tokenizer(examples['typed'], truncation=True, padding="max_length", max_length=128)

def preprocess_function(examples):
    tokenized_examples = tokenize_function(examples)
    tokenized_examples["labels"] = [label2id[label] for label in examples["device_type"]]
    return tokenized_examples

print("Tokenizing and preprocessing dataset...")
# Apply tokenization and label mapping to all splits
encoded_dataset = dataset.map(preprocess_function, batched=True)

# Set format for PyTorch
encoded_dataset = encoded_dataset.remove_columns([col for col in encoded_dataset['train'].column_names if col not in ['input_ids', 'attention_mask', 'labels']])
# Removing the line below to avoid issues with torchvision dependency in datasets formatting
# encoded_dataset.set_format("torch")

# --- 3. Define the model for sequence classification ---
print("Loading model...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# --- 4. Define training arguments and Trainer ---
print("Setting up training arguments...")
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1", # Use F1-score as the metric to choose the best model
    push_to_hub=False, # Set to True if you want to push to Hugging Face Hub
)

# Define metrics computation
def compute_metrics(p):
    predictions = np.argmax(p.predictions, axis=1)
    metric = evaluate.load("f1")
    f1_score = metric.compute(predictions=predictions, references=p.label_ids, average="weighted")["f1"]
    accuracy = accuracy_score(y_true=p.label_ids, y_pred=predictions)
    precision, recall, _, _ = precision_recall_fscore_support(y_true=p.label_ids, y_pred=predictions, average='weighted')
    return {
        "accuracy": accuracy,
        "f1": f1_score,
        "precision": precision,
        "recall": recall
    }

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    # Removed 'tokenizer=tokenizer' as it's not a valid argument for Trainer when dataset is already tokenized.
    compute_metrics=compute_metrics,
)

# --- 5. Train the model ---
print("Starting training...")
trainer.train()

# --- 6. Evaluate the model ---
print("Evaluating the trained model...")
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

print("Model fine-tuning complete!")

Preparing labels...
Detected labels: ['desktop', 'mobile', 'tablet']
Label to ID mapping: {'desktop': 0, 'mobile': 1, 'tablet': 2}
Tokenizing and preprocessing dataset...
Loading model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you

Setting up training arguments...
Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.761297,0.631757,0.628181,0.630120,0.631757
2,No log,0.681928,0.658784,0.655454,0.652162,0.658784
3,No log,0.731268,0.675676,0.670568,0.680045,0.675676
4,No log,0.741432,0.689189,0.680606,0.707856,0.689189
5,No log,0.736989,0.689189,0.684105,0.693902,0.689189


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Evaluating the trained model...


Evaluation results: {'eval_loss': 0.7369886636734009, 'eval_accuracy': 0.6891891891891891, 'eval_f1': 0.6841053068123132, 'eval_precision': 0.6939021532458723, 'eval_recall': 0.6891891891891891, 'eval_runtime': 2.6592, 'eval_samples_per_second': 111.313, 'eval_steps_per_second': 7.145, 'epoch': 5.0}
Model fine-tuning complete!


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [16]:
eval_results

{'eval_loss': 0.7369886636734009,
 'eval_accuracy': 0.6891891891891891,
 'eval_f1': 0.6841053068123132,
 'eval_precision': 0.6939021532458723,
 'eval_recall': 0.6891891891891891,
 'eval_runtime': 2.6592,
 'eval_samples_per_second': 111.313,
 'eval_steps_per_second': 7.145,
 'epoch': 5.0}